In [1]:
import toml 
import subprocess
import os
import shutil

In [2]:
def load_toml(path: str) -> dict:
    """Load config from toml file at path."""
    with open(path, "r", encoding="utf-8") as toml_file:
        data = toml.load(toml_file)
    return data
config_file = r"C:\ExploratoryAutomation\client_pag_exploratory\src\config.toml"
config = load_toml(config_file)
config

{'transcad_path': 'C:/Program Files/TransCAD 6.0',
 'r_program_path': 'C:/Program Files/R/R-4.4.2/bin/Rscript.exe',
 'moves4_path': 'C:/Users/Public/EPA/MOVES/MOVES4.0',
 'abm_model_path': 'C:/Network005/',
 'moves_input_excel_path': 'C:/ExploratoryAutomation/client_pag_exploratory/data/external/MOVES/',
 'out_dir': 'C:/ExploratoryAutomation/client_pag_exploratory/data/interim/',
 'src_dir': 'C:/ExploratoryAutomation/client_pag_exploratory/src/',
 'scenario_year': 2055,
 'scenario_name': 'Network005',
 'ev_share_percentage': 80}

In [4]:
#Copy ABM Outputs to the Processing Folder
src_VMT = os.path.join(config['abm_model_path'],"out","output_TDM_VMT.csv")
src_VHT = os.path.join(config['abm_model_path'],"out","output_TDM_VHT.csv")
dest_path = os.path.join(config['out_dir'])
shutil.copy2(src_VMT, dest_path)
shutil.copy2(src_VHT, dest_path)

'C:/ExploratoryAutomation/client_pag_exploratory/data/interim/output_TDM_VHT.csv'

In [5]:
results = subprocess.run(
    [
        "python",
        os.path.join(config['src_dir'], "create_moves_inputs.py")
    ],
    capture_output=True
)

ret_code = results.returncode

if ret_code != 0:
    raise Exception("ERROR: Step to create inputs using 'create_moves_inputs.py' script did not complete successfully!!!")

In [18]:
scenario_excel_file = os.path.join(config['out_dir'], str(config['scenario_year']) + "_moves4_" + config['scenario_name'] + ".xlsx")
#scenario_excel_file = scenario_excel_file.replace('/','\\')
moves_input_excel_file = os.path.join(config['moves_input_excel_path'], "moves4.xlsx")
shutil.copyfile(moves_input_excel_file, scenario_excel_file)

results = subprocess.run(
    [
        "python",
        os.path.join(config['src_dir'], "copy_outputs_to_excel.py"),
        scenario_excel_file,
        config['moves_input_excel_path'],
        config['out_dir'],
        str(config['scenario_year'])
    ],
    capture_output=True
)

ret_code = results.returncode



In [33]:
# update avft sheet
results = subprocess.run(
    [
        "python",
        os.path.join(config['src_dir'], "revise_avft.py"),
        str(config['ev_share_percentage']),
        str(2025),
        str(config['scenario_year']),
        config['moves_input_excel_path'],
        scenario_excel_file

    ],
    capture_output=True
)

ret_code = results.returncode

if ret_code != 0:
    raise Exception("ERROR: AVFT failed")

In [34]:
# Write input database XML file 
# Rscript.exe write_moves_input_database_xml.R scenario_year, scenario_name, input_excel_file, output_dir

scenario_xml_file = os.path.join(config['out_dir'], str(config['scenario_year']) + "_moves4_" + config['scenario_name'] + ".mrs")
#scenario_xml_file = scenario_xml_file.replace('/','\\')
print(scenario_excel_file)
print(scenario_xml_file)
results = subprocess.run(
    [
        config['r_program_path'],
        os.path.join(config['src_dir'], "write_moves_input_database_xml.R"),
        str(config['scenario_year']),
        config['scenario_name'],
        scenario_excel_file,
        scenario_xml_file
    ],
    capture_output=True
)

C:/ExploratoryAutomation/client_pag_exploratory/data/interim/2055_moves4_Network005.xlsx
C:/ExploratoryAutomation/client_pag_exploratory/data/interim/2055_moves4_Network005.mrs


In [35]:
# Write run specification file
# Rscript.exe write_moves_run_spec.R scenario_year, scenario_name, output_dir

scenario_spec_file = os.path.join(config['out_dir'], str(config['scenario_year']) + "_moves4_" + config['scenario_name'] + ".xml")
#scenario_spec_file = scenario_spec_file.replace('/','\\')

results = subprocess.run(
    [
        config['r_program_path'],
        os.path.join(config['src_dir'], "write_moves_run_spec.R"),
        str(config['scenario_year']),
        config['scenario_name'],
        scenario_spec_file
    ],
    capture_output=True
)

In [37]:
def update_path_in_batch_file(file_path, key, value):
    # Read the file
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Iterate through each line
    for i, line in enumerate(lines):
        # Check if the line starts with the "set key="
        if "set " + key + '=' in line:
            lines[i] = "set " + key + '=' + value + '\n'

    # Write back to the file
    with open(file_path, 'w') as file:
        file.writelines(lines)

In [38]:
# Run MOVES from a batch file

moves_batch_file = os.path.join(config['src_dir'], "run_moves.bat")

update_path_in_batch_file(moves_batch_file, "moves_dir", config['moves4_path'])
update_path_in_batch_file(moves_batch_file, "input_xml_file", scenario_xml_file)
update_path_in_batch_file(moves_batch_file, "input_spec_file", scenario_spec_file)

In [42]:
results = subprocess.run(moves_batch_file, capture_output=True, shell=True)

ret_code = results.returncode

if ret_code != 0:
    raise Exception("ERROR: MOVES run did not finish successfully!!!")

In [43]:
# Process outputs
# Rscript.exe process_moves_outputs.R scenario_year, scenario_name, output_dir, database_password
results = subprocess.run(
    [
        config['r_program_path'],
        os.path.join(config['src_dir'], "process_moves_outputs.R"),
        str(config['scenario_year']),
        config['scenario_name'],
        os.path.join(config['abm_model_path'], "out").replace('/','\\'),
        "PAG531m0ves*"
    ],
    capture_output=True
)